# Stage 8.1 — Pilot Training Task 1: Plant Species Identification (DaunKu AI)

**Lingkungan Target:** Google Colab GPU (Runtime -> Change runtime type -> T4 GPU / V100 / L4)  
**Model Pilot:** `EfficientNet-B0` (pretrained ImageNet-1K, 68 kelas single-species)  
**Durasi Pilot:** Maksimal **3 epoch** (untuk validasi pipeline, bukan model final)  

> **Aturan Ketat Stage 8.1:**
> - Hanya menggunakan **Train Stage 6** untuk 68 kelas single-species (**37.408 gambar**), dibagi menjadi `inner_train` (~85%) dan `inner_val` (~15%) berdasarkan `meta_component_id` (`seed = 42`).
> - **Stage 6 Validation (6.068 gambar)** tidak digunakan pada pilot ini.
> - **Locked Test (6.047 gambar)** **DILARANG** dibaca atau dievaluasi pada Stage 8.1.
> - Dataset asli bersifat **READ-ONLY** dan dibaca bertahap melalui `DataLoader` (tidak dimuat seluruhnya ke RAM).


## Cell 1 — Pemeriksaan GPU dan Informasi Runtime


In [ ]:
import os
import platform
import sys
import torch
import torchvision

print("=== INFORMASI RUNTIME & HARDWARE ===")
print(f"Python       : {sys.version.split()[0]} ({platform.platform()})")
print(f"PyTorch      : {torch.__version__}")
print(f"Torchvision  : {torchvision.__version__}")
print(f"CUDA Avail   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    vram_gb = gpu_props.total_memory / (1024 ** 3)
    print(f"GPU Device   : {gpu_name}")
    print(f"GPU VRAM     : {vram_gb:.2f} GB")
    DEVICE = torch.device("cuda")
else:
    gpu_name = "NONE (CPU ONLY)"
    vram_gb = 0.0
    DEVICE = torch.device("cpu")
    raise RuntimeError(
        "GPU TIDAK TERDETEKSI! Aktifkan GPU terlebih dahulu di Google Colab: "
        "Menu Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4/L4/V100)."
    )


## Cell 2 — Instalasi Dependency yang Diperlukan


In [ ]:
# Memasang dependency ringan tanpa menimpa PyTorch/Torchvision CUDA bawaan Colab
!pip -q install pyyaml scikit-learn pandas pillow tqdm

import pandas as pd
import numpy as np
import sklearn
import PIL
import yaml

print("=== VERSI DEPENDENCY UTAMA ===")
print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"Pillow       : {PIL.__version__}")
print(f"PyYAML       : {yaml.__version__}")


## Cell 3 — Mount Google Drive untuk Menyimpan Checkpoint & Log


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = True  # Ubah ke False hanya jika tidak memakai Google Drive

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE_DIR = Path("/content/drive/MyDrive/DaunKu_AI")
    DRIVE_OUTPUT_DIR = DRIVE_BASE_DIR / "stage8_1_pilot"
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Folder output permanen: {DRIVE_OUTPUT_DIR}")
else:
    DRIVE_OUTPUT_DIR = Path("/content/stage8_1_pilot_outputs")
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Output lokal sementara: {DRIVE_OUTPUT_DIR}")


## Cell 4 — Mengakses Repository `DaunKu_AI`


In [ ]:
import subprocess

REPO_URL = "https://github.com/nirvanaguys/DaunKu_AI.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/DaunKu_AI")

# Opsi: Jika artefak Stage 6/8.1 belum di-push ke GitHub, Anda dapat menunjuk langsung ke salinan repo di Google Drive:
# Contoh: CUSTOM_REPO_DIR = Path("/content/drive/MyDrive/DaunKu_AI/daunku_ai")
CUSTOM_REPO_DIR = None

if CUSTOM_REPO_DIR is not None and Path(CUSTOM_REPO_DIR).exists():
    PROJECT_ROOT = Path(CUSTOM_REPO_DIR)
    print(f"Menggunakan direktori proyek dari Google Drive: {PROJECT_ROOT}")
else:
    if not REPO_DIR.exists():
        print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
        subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        print(f"Repository sudah ada di {REPO_DIR}, melakukan git pull...")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
    PROJECT_ROOT = REPO_DIR

print(f"PROJECT_ROOT: {PROJECT_ROOT}")


## Cell 5 — Konfigurasi Path Dataset (Mudah Diubah Pengguna)

> **Catatan:** Dataset mentah (~8,8 GB) tidak disimpan di GitHub. Pastikan folder `Dataset/` sudah tersedia di Google Drive atau runtime lokal Colab (misal `/content/drive/MyDrive/DaunKu_AI/Dataset` atau hasil ekstrak zip ke `/content/Dataset` untuk pembacaan I/O yang jauh lebih cepat).


In [ ]:
# === KONFIGURASI PATH DATASET (SESUAIKAN DENGAN LOKASI DATASET ANDA) ===
DATASET_DIR = Path("/content/drive/MyDrive/DaunKu_AI/Dataset")
# Tips kecepatan I/O Colab: jika Anda menyimpan Dataset.zip di Drive, ekstrak ke /content/Dataset lalu ubah DATASET_DIR = Path("/content/Dataset")

TRAIN_MANIFEST_PATH = PROJECT_ROOT / "artifacts" / "splits" / "train_manifest.csv"
LABEL_MAPPING_PATH = PROJECT_ROOT / "configs" / "label_mapping.yaml"

# Validasi keberadaan folder Dataset dan manifest
if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Folder Dataset TIDAK DITEMUKAN di: {DATASET_DIR}\n"
        "INSTRUKSI PENYIAPAN:\n"
        "1. Unggah folder 'Dataset/' dari laptop Anda (daunku_ai/Dataset/) ke Google Drive Anda, misal di:\n"
        "   MyDrive/DaunKu_AI/Dataset/\n"
        "2. Pastikan subfolder dataset (contoh: 'Arabian Jasmine Leaf Condition Dataset (AJLCD-2025', "
        "'goletplant.v5i.tensorflow', 'Leaf Image Dataset for Plant disease Classificatio', dll) berada langsung di dalam folder tersebut.\n"
        "3. Sesuaikan variabel DATASET_DIR di Cell 5 lalu jalankan ulang cell ini."
    )

if not TRAIN_MANIFEST_PATH.exists() or not LABEL_MAPPING_PATH.exists():
    raise FileNotFoundError(
        f"Artefak Stage 5/6 tidak ditemukan di {PROJECT_ROOT}.\n"
        f"Pastikan file berikut tersedia:\n"
        f" - {TRAIN_MANIFEST_PATH}\n"
        f" - {LABEL_MAPPING_PATH}\n"
        "Jika artefak belum di-push ke GitHub, unggah folder 'daunku_ai/' ke Google Drive lalu atur CUSTOM_REPO_DIR di Cell 4."
    )

print(f"DATASET_DIR terdeteksi : {DATASET_DIR}")
print(f"Subfolder ditemukan    : {len([p for p in DATASET_DIR.iterdir() if p.is_dir()])} folder")


## Cell 6 & 7 — Membaca Manifest, Verifikasi 68 Single-Species, dan Validasi Path & Keterbacaan Gambar


In [ ]:
from PIL import Image

EXPERIMENT_SEED = 42
PIL_VERIFY_SAMPLE = 200  # Uji buka 200 sampel gambar acak (tidak memuat seluruh dataset ke RAM)

tm = pd.read_csv(TRAIN_MANIFEST_PATH, low_memory=False)
with open(LABEL_MAPPING_PATH, "r", encoding="utf-8") as f:
    label_mapping = yaml.safe_load(f)

# Filter HANYA data Train Stage 6 untuk Task 1 (tanpa membaca metadata Stage 6 Validation maupun Locked Test)
train_sp = tm[tm["split_species_id"] == "train"].copy()
is_combo = train_sp["plant_species"].astype(str).str.contains(" / ", regex=False)
train_s68 = train_sp[~is_combo].copy()
combo_train_df = train_sp[is_combo].copy()

n_train = len(train_s68)
n_combo_train = len(combo_train_df)
n_classes = int(train_s68["plant_species"].nunique())

print("=== VERIFIKASI DATA TRAIN TASK 1 (DARI train_manifest.csv) ===")
print(f"Train 68 single-species : {n_train} gambar ({n_classes} kelas)")
print(f"Train 37 combo excluded : {n_combo_train} gambar ({combo_train_df['plant_species'].nunique()} kelas)")
print("Metadata Stage 6 Validation & Locked Test TIDAK DIBACA oleh notebook ini.")

assert n_classes == 68, f"Harus 68 kelas, ditemukan {n_classes}"
assert n_train == 37408, f"Harus 37.408 gambar train single-species, ditemukan {n_train}"
assert n_combo_train == 92, f"Harus 92 gambar train combo yang dieksklusikan, ditemukan {n_combo_train}"

# Verifikasi konsistensi terhadap configs/label_mapping.yaml
lm_valid_single = {
    e["plant_species"]
    for e in label_mapping["label_mappings"]
    if e["task_eligibility"]["species_id"]
    and e["usage_status"] == "included"
    and " / " not in str(e["plant_species"])
}
assert set(train_s68["plant_species"].unique()) == lm_valid_single, "Ketidaksesuaian manifest vs label_mapping.yaml!"
assert not train_s68["plant_species"].astype(str).str.contains(" / ", regex=False).any()
assert not (train_s68["plant_species"].astype(str).str.lower() == "unknown").any()
assert not (train_s68["usage_status"].astype(str) == "quarantined").any()
assert (train_s68["split_species_id"] == "train").all()

# Mapping kelas deterministik 0..67
SORTED_CLASSES = sorted(lm_valid_single)
CLASS_TO_IDX = {c: i for i, c in enumerate(SORTED_CLASSES)}
IDX_TO_CLASS = {i: c for i, c in enumerate(SORTED_CLASSES)}
train_s68["class_idx"] = train_s68["plant_species"].map(CLASS_TO_IDX)

# Validasi keberadaan seluruh 37.408 path gambar (konversi backslash Windows -> slash Linux untuk Colab)
train_s68["abs_path"] = train_s68["relative_path"].astype(str).apply(
    lambda p: str(DATASET_DIR / p.replace("\\", "/"))
)
missing = [p for p in train_s68["abs_path"] if not os.path.isfile(p)]
print(f"[Path Check — Penuh] Seluruh {len(train_s68)} path diperiksa: Ditemukan={len(train_s68) - len(missing)}, Hilang={len(missing)}")
if missing:
    raise FileNotFoundError(f"Ada {len(missing)} file gambar tidak ditemukan! Contoh: {missing[:5]}")

# Uji keterbacaan PIL HANYA pada 200 sampel acak (bukan seluruh 37.408 gambar)
corrupt = []
for p in train_s68["abs_path"].sample(n=min(PIL_VERIFY_SAMPLE, len(train_s68)), random_state=EXPERIMENT_SEED):
    try:
        with Image.open(p) as im:
            im.convert("RGB")
    except Exception as exc:
        corrupt.append((p, str(exc)))
print(f"[PIL Readability — Sampel] Hanya {PIL_VERIFY_SAMPLE} sampel acak yang diuji buka dengan PIL (bukan seluruh {len(train_s68)} gambar): OK={PIL_VERIFY_SAMPLE - len(corrupt)}, Corrupt={len(corrupt)}")
if corrupt:
    raise RuntimeError(f"Ditemukan gambar corrupt pada sampel PIL: {corrupt[:5]}")
print("VALIDASI SELESAI: seluruh 37.408 path terverifikasi ada & 200/200 sampel PIL dapat dibaca.")


## Cell 8 — Membuat `inner_train` (~85%) dan `inner_val` (~15%) Berbasis `meta_component_id`


In [ ]:
import json
import random
from collections import Counter, defaultdict

def greedy_stratified_group_split(df, group_col="meta_component_id", label_col="plant_species", val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    group_labels = defaultdict(Counter)
    for g, lbl in zip(df[group_col].astype(str), df[label_col].astype(str)):
        group_labels[g][lbl] += 1

    total_class_counts = Counter(df[label_col].astype(str))
    target_val_counts = {c: max(1.0, cnt * val_ratio) for c, cnt in total_class_counts.items()}

    class_to_groups = defaultdict(list)
    for g, cnts in group_labels.items():
        primary_lbl = cnts.most_common(1)[0][0]
        class_to_groups[primary_lbl].append(g)

    for lbl in sorted(class_to_groups.keys()):
        class_to_groups[lbl].sort()
        rng.shuffle(class_to_groups[lbl])

    val_groups, train_groups = set(), set()
    current_val_counts = Counter()
    ordered_classes = sorted(class_to_groups.keys(), key=lambda c: (len(class_to_groups[c]), c))

    for lbl in ordered_classes:
        groups = class_to_groups[lbl]
        if len(groups) < 2:
            raise ValueError(f"Kelas '{lbl}' hanya punya {len(groups)} meta_component_id; tidak bisa dibagi!")
        assigned_val, assigned_train = 0, 0
        for idx, g in enumerate(groups):
            if g in val_groups or g in train_groups:
                continue
            remaining = len(groups) - idx
            g_size = sum(group_labels[g].values())
            if assigned_train == 0 and remaining == 1:
                train_groups.add(g); assigned_train += 1
            elif assigned_val == 0 and remaining >= 2:
                val_groups.add(g); current_val_counts.update(group_labels[g]); assigned_val += 1
            elif (current_val_counts[lbl] + g_size <= target_val_counts[lbl]) and (assigned_train > 0 or remaining > 1):
                val_groups.add(g); current_val_counts.update(group_labels[g]); assigned_val += 1
            else:
                train_groups.add(g); assigned_train += 1
    return train_groups, val_groups

tr_groups, va_groups = greedy_stratified_group_split(train_s68, val_ratio=0.15, seed=EXPERIMENT_SEED)
inner_train_df = train_s68[train_s68["meta_component_id"].astype(str).isin(tr_groups)].reset_index(drop=True)
inner_val_df = train_s68[train_s68["meta_component_id"].astype(str).isin(va_groups)].reset_index(drop=True)

# Assert zero group leakage & full 68-class coverage
overlap_mc = set(inner_train_df["meta_component_id"]) & set(inner_val_df["meta_component_id"])
assert len(overlap_mc) == 0, f"LEAKAGE! {len(overlap_mc)} meta_component_id overlap!"
assert len(inner_train_df) + len(inner_val_df) == 37408
assert inner_train_df["plant_species"].nunique() == 68
assert inner_val_df["plant_species"].nunique() == 68

# Simpan manifest inner_train & inner_val ke Drive agar reproducible
save_cols = ["image_id", "relative_path", "source_dataset", "plant_species", "class_idx", "parent_source_group_id", "meta_component_id", "split_species_id"]
inner_train_df[save_cols].to_csv(DRIVE_OUTPUT_DIR / "task1_inner_train.csv", index=False)
inner_val_df[save_cols].to_csv(DRIVE_OUTPUT_DIR / "task1_inner_val.csv", index=False)
with open(DRIVE_OUTPUT_DIR / "task1_class_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"num_classes": 68, "class_to_idx": CLASS_TO_IDX, "idx_to_class": {str(k): v for k, v in IDX_TO_CLASS.items()}}, f, indent=2, ensure_ascii=False)

print("=== RINGKASAN INNER SPLIT (HANYA DARI TRAIN STAGE 6) ===")
print(f"inner_train : {len(inner_train_df)} gambar ({len(inner_train_df)/37408*100:.2f}%) | {inner_train_df['meta_component_id'].nunique()} meta_components | {inner_train_df['plant_species'].nunique()} kelas")
print(f"inner_val   : {len(inner_val_df)} gambar ({len(inner_val_df)/37408*100:.2f}%) | {inner_val_df['meta_component_id'].nunique()} meta_components | {inner_val_df['plant_species'].nunique()} kelas")
print(f"Overlap meta_component_id : {len(overlap_mc)} (ZERO GROUP LEAKAGE)")
print("Stage 6 Validation (6.068) & Locked Test (6.047) TIDAK DIGUNAKAN.")


## Cell 9 — Membuat PyTorch Dataset (Streaming dari Disk) & DataLoader


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(EXPERIMENT_SEED)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Augmentasi HANYA pada inner_train
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Deterministik untuk inner_val (tanpa augmentasi acak)
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class DaunKuSpeciesDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.paths = df["abs_path"].tolist()
        self.labels = df["class_idx"].astype(int).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as img:
            img = img.convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, self.labels[idx]

# Penyesuaian batch size berdasarkan VRAM GPU aktual
BATCH_SIZE = 64 if vram_gb >= 10.0 else 32
NUM_WORKERS = 2

train_dataset = DaunKuSpeciesDataset(inner_train_df, transform=train_transform)
val_dataset = DaunKuSpeciesDataset(inner_val_df, transform=val_transform)

g_loader = torch.Generator()
g_loader.manual_seed(EXPERIMENT_SEED)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, generator=g_loader
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"DataLoader siap | BATCH_SIZE={BATCH_SIZE} | train_batches={len(train_loader)} | val_batches={len(val_loader)}")


## Cell 10 — Inisialisasi Model `EfficientNet-B0`, Loss, Optimizer, & Scheduler

> **Catatan Scheduler untuk Pilot 3 Epoch:**  
> Pada rencana *full training* (25–30 epoch), scheduler menggunakan *Linear Warmup* 3 epoch diikuti *Cosine Annealing*. Namun pada **pilot maksimal 3 epoch**, warmup 3 epoch penuh tidak akan pernah memasuki fase cosine decay. Oleh karena itu, khusus untuk pilot 3 epoch ini digunakan **1 epoch linear warmup + 2 epoch cosine annealing** (tanpa memperpanjang durasi pilot melewati 3 epoch).


In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

MAX_EPOCHS = 3
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
PILOT_WARMUP_EPOCHS = 1
MIN_LR = 1e-6

model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 68)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Scheduler konsisten untuk pilot 3 epoch: 1 epoch linear warmup + 2 epoch cosine decay
warmup_sched = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=PILOT_WARMUP_EPOCHS
)
cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, MAX_EPOCHS - PILOT_WARMUP_EPOCHS), eta_min=MIN_LR
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[PILOT_WARMUP_EPOCHS]
)
scaler = torch.amp.GradScaler("cuda")

total_params = sum(p.numel() for p in model.parameters())
print(f"Model EfficientNet-B0 siap di {DEVICE} | Total parameter: {total_params:,} | Output classes: 68")


## Cell 11 — Pilot Training Maksimal 3 Epoch (Tanpa Menyentuh Locked Test)


In [ ]:
import time
from sklearn.metrics import f1_score, precision_recall_fscore_support
from tqdm.auto import tqdm

def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    top1_correct = 0
    top3_correct = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Inner Val", leave=False):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            with torch.amp.autocast("cuda"):
                logits = model(images)
                loss = criterion(logits, targets)

            bs = targets.size(0)
            total_loss += loss.item() * bs
            total_samples += bs

            _, pred_top3 = logits.topk(3, dim=1, largest=True, sorted=True)
            top1_correct += (pred_top3[:, 0] == targets).sum().item()
            top3_correct += (pred_top3 == targets.unsqueeze(1)).any(dim=1).sum().item()

            all_preds.extend(pred_top3[:, 0].cpu().numpy().tolist())
            all_targets.extend(targets.cpu().numpy().tolist())

    avg_loss = total_loss / total_samples
    top1_acc = top1_correct / total_samples
    top3_acc = top3_correct / total_samples
    macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    prec, rec, f1_cls, sup = precision_recall_fscore_support(
        all_targets, all_preds, labels=list(range(68)), zero_division=0
    )
    per_class_df = pd.DataFrame({
        "class_idx": list(range(68)),
        "plant_species": [IDX_TO_CLASS[i] for i in range(68)],
        "precision": prec,
        "recall": rec,
        "f1_score": f1_cls,
        "support": sup,
    })
    return {
        "val_loss": avg_loss,
        "val_acc": top1_acc,
        "top1_acc": top1_acc,
        "top3_acc": top3_acc,
        "macro_f1": macro_f1,
    }, per_class_df

history = []
best_macro_f1 = -1.0
best_per_class_df = None

LOCAL_CKPT_DIR = PROJECT_ROOT / "models"
LOCAL_LOG_DIR = PROJECT_ROOT / "reports" / "training_experiments"
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT_PATH = LOCAL_CKPT_DIR / "stage8_task1_pilot_efficientnet_b0_best.pt"

for epoch in range(1, MAX_EPOCHS + 1):
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    model.train()
    running_loss = 0.0
    seen = 0
    current_lr = optimizer.param_groups[0]["lr"]

    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS} [Train]", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = targets.size(0)
        running_loss += loss.item() * bs
        seen += bs

    scheduler.step()
    train_loss = running_loss / seen

    val_metrics, per_class_df = evaluate_epoch(model, val_loader, criterion, DEVICE)
    epoch_sec = time.time() - t0
    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    record = {
        "epoch": epoch,
        "lr": current_lr,
        "train_loss": round(train_loss, 5),
        "inner_val_loss": round(val_metrics["val_loss"], 5),
        "inner_val_acc": round(val_metrics["val_acc"], 5),
        "top1_acc": round(val_metrics["top1_acc"], 5),
        "top3_acc": round(val_metrics["top3_acc"], 5),
        "macro_f1": round(val_metrics["macro_f1"], 5),
        "epoch_time_sec": round(epoch_sec, 2),
        "peak_gpu_memory_mb": round(peak_vram_mb, 2),
    }
    history.append(record)

    print(
        f"Epoch {epoch}/{MAX_EPOCHS} | lr={current_lr:.2e} | "
        f"train_loss={record['train_loss']:.4f} | val_loss={record['inner_val_loss']:.4f} | "
        f"top1={record['top1_acc']:.4f} | top3={record['top3_acc']:.4f} | "
        f"macro_f1={record['macro_f1']:.4f} | time={record['epoch_time_sec']:.1f}s | "
        f"peak_vram={record['peak_gpu_memory_mb']:.0f} MB"
    )

    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]
        best_per_class_df = per_class_df.copy()
        torch.save(
            {
                "epoch": epoch,
                "architecture": "efficientnet_b0",
                "num_classes": 68,
                "class_to_idx": CLASS_TO_IDX,
                "model_state_dict": model.state_dict(),
                "best_inner_val_macro_f1": best_macro_f1,
                "pilot_only": True,
            },
            BEST_CKPT_PATH,
        )


## Cell 12 — Menyimpan Checkpoint, Log, Metrik Per-Kelas, dan Konfigurasi Aktual ke Drive


In [ ]:
import shutil

actual_config = {
    "stage": "8.1_task1_pilot",
    "model": "efficientnet_b0",
    "pretrained": "IMAGENET1K_V1",
    "num_classes": 68,
    "input_size": 224,
    "max_epochs": MAX_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "scheduler": f"LinearWarmup({PILOT_WARMUP_EPOCHS}ep) + CosineAnnealing({MAX_EPOCHS - PILOT_WARMUP_EPOCHS}ep)",
    "experiment_seed": EXPERIMENT_SEED,
    "gpu_name": gpu_name,
    "gpu_vram_gb": round(vram_gb, 2),
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "inner_train_images": len(inner_train_df),
    "inner_val_images": len(inner_val_df),
    "stage6_validation_used": False,
    "locked_test_used": False,
}

metrics_payload = {
    "config": actual_config,
    "best_inner_val_macro_f1": round(best_macro_f1, 5),
    "epochs": history,
    "note": "Hasil pilot 3 epoch hanya untuk validasi pipeline training, BUKAN performa final model.",
}

metrics_json_path = LOCAL_LOG_DIR / "stage8_task1_pilot_metrics.json"
per_class_csv_path = LOCAL_LOG_DIR / "stage8_task1_pilot_per_class.csv"
actual_cfg_path = LOCAL_LOG_DIR / "stage8_task1_pilot_actual_config.yaml"

with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)
if best_per_class_df is not None:
    best_per_class_df.to_csv(per_class_csv_path, index=False)
with open(actual_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(actual_config, f, sort_keys=False, allow_unicode=True)

# Salin seluruh artefak pilot ke Google Drive agar persisten
for src in [BEST_CKPT_PATH, metrics_json_path, per_class_csv_path, actual_cfg_path]:
    if src.exists():
        shutil.copy2(src, DRIVE_OUTPUT_DIR / src.name)

print("Checkpoint & log tersimpan secara lokal dan disalin ke Google Drive:")
print(f" - Checkpoint : {DRIVE_OUTPUT_DIR / BEST_CKPT_PATH.name}")
print(f" - Metrics    : {DRIVE_OUTPUT_DIR / metrics_json_path.name}")
print(f" - Per-Class  : {DRIVE_OUTPUT_DIR / per_class_csv_path.name}")
print(f" - Config     : {DRIVE_OUTPUT_DIR / actual_cfg_path.name}")


## Cell 13 — Ringkasan Hasil Pilot Stage 8.1


In [ ]:
history_df = pd.DataFrame(history)
print("=== RINGKASAN METRIK PILOT (MAKSIMAL 3 EPOCH) ===")
display(history_df)

if best_per_class_df is not None:
    print("\n=== 5 KELAS DENGAN F1 TERENDAH PADA INNER_VAL (DIAGNOSTIK AWAL) ===")
    display(best_per_class_df.sort_values("f1_score").head(5))

print("\nKONFIRMASI KEPATUHAN:")
print(" - Stage 6 Validation (6.068 gambar): TIDAK DIGUNAKAN")
print(" - Locked Test (6.047 gambar)       : TIDAK DIGUNAKAN")
print(" - Catatan: Ini adalah hasil pilot 3 epoch (bukan klaim performa final).")
